# AquiferGuard — Groundwater Uranium Risk Classification

This notebook prepares groundwater data, explores uranium concentrations, trains a **Decision Tree Classifier**, evaluates it, and saves the complete prediction pipeline.

**Project classification rule:** uranium concentration greater than `0.03 mg/L` is labelled **Elevated**; values at or below this threshold are labelled **Not Elevated**. This is the project's modelling threshold, not a substitute for official water-quality assessment or laboratory testing.

Run the notebook from top to bottom. Keep it inside the project's `notebooks` folder.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../dataset/raw/cgwb_punjab_groundwater_quality_raw.csv")

print("Dataset shape:", df.shape)
display(df.head())

## 2. Inspect the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values:")
print(df.isnull().sum())

print("\\nStatistical summary:")
display(df.describe())

## 3. Select the columns used in this project

In [ ]:
df = df[[
    "Station",
    "District",
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Bicarbonate (mg/L)",
    "Chloride (mg/L)",
    "Sodium (mg/L)",
    "Uranium(mg/L)"
]].copy()

print("Selected columns:", df.columns.tolist())
display(df.head())

## 4. Remove records without uranium measurements and remove duplicates

In [ ]:
df = df.dropna(subset=["Uranium(mg/L)"])
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)
print("Missing uranium values:", df["Uranium(mg/L)"].isnull().sum())

## 5. Create the uranium risk category

In [ ]:
# 1 = Elevated, 0 = Not Elevated
df["Uranium_Risk"] = (df["Uranium(mg/L)"] > 0.03).astype(int)

df["Risk_Category"] = df["Uranium_Risk"].map({
    0: "Not Elevated",
    1: "Elevated"
})

print("Risk category counts:")
print(df["Risk_Category"].value_counts())

display(df[["Uranium(mg/L)", "Uranium_Risk", "Risk_Category"]].head())

## 6. Explore the uranium risk distribution

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Risk_Category")
plt.title("Groundwater Uranium Risk Distribution")
plt.xlabel("Uranium Risk Category")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()

## 7. Explore uranium concentration

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Uranium(mg/L)", bins=30, kde=True)
plt.axvline(0.03, color="red", linestyle="--", label="Project threshold: 0.03 mg/L")
plt.title("Distribution of Uranium Concentration")
plt.xlabel("Uranium (mg/L)")
plt.ylabel("Number of Samples")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Prepare the input features

In [ ]:
features = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Bicarbonate (mg/L)",
    "Chloride (mg/L)",
    "Sodium (mg/L)"
]

# Invalid pH values are treated as missing
ph_column = "Potential of Hydrogen (pH)"
df.loc[~df[ph_column].between(0, 14), ph_column] = pd.NA

print("Missing values in model features:")
print(df[features].isnull().sum())

## 9. Save the prepared dataset

In [ ]:
# Save prepared data. Missing feature values will be imputed by the model pipeline.
df.to_csv("../dataset/processed/aquiferguard_cleaned.csv", index=False)

print("Prepared dataset saved successfully.")

## 10. Split the data into training and testing sets

**Important improvement:** split the data before calculating medians. This prevents information from the test set from influencing the training process.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[features]
y = df["Uranium_Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 11. Build and train the Decision Tree model

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# The imputer learns medians from training data only
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Decision Tree model pipeline trained successfully.")

## 12. Evaluate the model

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))

print("\\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Elevated", "Elevated"],
    zero_division=0
))

print("Training accuracy:", round(model.score(X_train, y_train) * 100, 2), "%")
print("Testing accuracy:", round(model.score(X_test, y_test) * 100, 2), "%")

## 13. Display the confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Elevated", "Elevated"],
    yticklabels=["Not Elevated", "Elevated"]
)
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.title("AquiferGuard — Confusion Matrix")
plt.tight_layout()
plt.show()

## 14. Visualise the trained Decision Tree

In [ ]:
from sklearn.tree import plot_tree

tree_model = model.named_steps["classifier"]

plt.figure(figsize=(18, 10))
plot_tree(
    tree_model,
    feature_names=features,
    class_names=["Not Elevated", "Elevated"],
    filled=True,
    rounded=True
)
plt.title("AquiferGuard — Decision Tree Classifier")
plt.tight_layout()
plt.show()

## 15. Save the complete model pipeline

In [ ]:
import joblib

joblib.dump(model, "../models/aquiferguard_model.pkl")

print("Final AquiferGuard model pipeline saved successfully.")

## 16. Compare actual and predicted categories for sample records

In [ ]:
sample_data = X_test.head(5)
sample_predictions = model.predict(sample_data)

sample_results = pd.DataFrame({
    "Actual Risk": y_test.head(5).values,
    "Predicted Risk": sample_predictions
})

sample_results["Actual Category"] = sample_results["Actual Risk"].map({
    0: "Not Elevated",
    1: "Elevated"
})
sample_results["Predicted Category"] = sample_results["Predicted Risk"].map({
    0: "Not Elevated",
    1: "Elevated"
})

display(sample_results)

---

**End of notebook.** The complete model pipeline is saved at `models/aquiferguard_model.pkl`, and the prepared dataset is saved at `dataset/processed/aquiferguard_cleaned.csv`.

**Why the pipeline matters:** it applies median imputation using medians learned only from training data. The same saved pipeline also performs this preprocessing when the Streamlit application makes predictions.

*Reminder:* AquiferGuard provides an academic model estimate. It does not establish whether a water source is safe to drink.